In [9]:
import json
import pandas as pd

train_path = ["/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_1.jsonl",
        "/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_2.jsonl",
        "/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_3.jsonl",
        "/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_4.jsonl",
        "/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_5.jsonl"]

test_path = ["/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/test_features.jsonl"]

def load_json_lines(paths, max_lines=None):
    data = []
    
    for path in paths: 
        print(path)
        with open(path, 'r') as f:
            for i, line in enumerate(f):
                if max_lines and i >= max_lines:
                    break
                data.append(json.loads(line))
    
    return pd.DataFrame(data)


train_df = load_json_lines(train_path,50000)
test_df  = load_json_lines(test_path,10000)


/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_1.jsonl
/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_2.jsonl
/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_3.jsonl
/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_4.jsonl
/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/train_features_5.jsonl
/Users/metecemturan/Documents/code/malware_detection/extracred/ember2018/test_features.jsonl


In [10]:
import numpy as np

def extract_features(df):
    features = []
    
    for _, row in df.iterrows():
        feat = []
        feat.extend(row["histogram"])
        feat.extend(row["byteentropy"])
        features.append(feat)
    
    return np.array(features)

# ----------- Train set -----------

train_df = train_df[train_df["label"] != -1]

X_train = extract_features(train_df)
y_train = train_df["label"].values

print("After cleaning train:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# ----------- Test set -----------
test_df = test_df[test_df["label"] != -1]

X_test = extract_features(test_df)
y_test = test_df["label"].values

print("\nAfter cleaning test:")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

After cleaning train:
X_train shape: (183033, 512)
y_train shape: (183033,)

After cleaning test:
X_test shape: (10000, 512)
y_test shape: (10000,)


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

model = RandomForestClassifier(
    n_estimators=300,       
    max_depth=40,            
    min_samples_split=5,     
    min_samples_leaf=2,      
    max_features="sqrt",   
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# Eğit
model.fit(X_train, y_train)

# Test
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

# Classification report (3 decimal)
report_dict = classification_report(y_test, y_pred, output_dict=True)
print("\nClassification Report:")
for key in report_dict.keys():
    if isinstance(report_dict[key], dict):
        report_dict[key] = {k: round(v,3) for k,v in report_dict[key].items()}
    else:
        report_dict[key] = round(report_dict[key],3)

print(report_dict)

# ROC AUC
roc_auc = round(roc_auc_score(y_test, y_prob),3)
print("\nROC AUC Score:", roc_auc)


Classification Report:
{'0': {'precision': 0.914, 'recall': 0.893, 'f1-score': 0.903, 'support': 5006.0}, '1': {'precision': 0.895, 'recall': 0.916, 'f1-score': 0.905, 'support': 4994.0}, 'accuracy': 0.904, 'macro avg': {'precision': 0.905, 'recall': 0.904, 'f1-score': 0.904, 'support': 10000.0}, 'weighted avg': {'precision': 0.905, 'recall': 0.904, 'f1-score': 0.904, 'support': 10000.0}}

ROC AUC Score: 0.968


In [12]:
import joblib

model_filename = "ember_model.pkl"
joblib.dump(model, model_filename)
print(f"Model kaydedildi: {model_filename}")

# Modeli tekrar yüklemek istersen:
# loaded_model = joblib.load(model_filename)

Model kaydedildi: ember_model.pkl


In [13]:
# import lightgbm as lgb
# model = lgb.LGBMClassifier(
#     n_estimators=500, 
#     learning_rate=0.05, 
#     n_jobs=-1, 
#     random_state=42
# )
# model.fit(X_train, y_train)
# from sklearn.metrics import classification_report, roc_auc_score
# from pprint import pprint

# # Tahmin
# y_pred = model.predict(X_test)
# y_prob = model.predict_proba(X_test)[:,1]

# # Classification report
# report = classification_report(y_test, y_pred, output_dict=True)
# print("\nLightGBM Classification Report:")
# pprint(report)

# # ROC AUC
# roc_auc = roc_auc_score(y_test, y_prob)
# print("\nLightGBM ROC AUC Score:", roc_auc)